In [2]:
import pandas as pd
from scipy.datasets import download_all
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from imblearn.over_sampling import RandomOverSampler
from collections import Counter
import tensorflow as tf
import random
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.regularizers import l2
from sklearn.utils import class_weight
from scikeras.wrappers import KerasClassifier
import joblib
from sklearn.metrics import classification_report, confusion_matrix, f1_score

2026-06-07 14:37:37.874663: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Data Preprocessing

First, we need to clean up the raw data. In this step, we will:
1. **Drop irrelevant identifiers:** `user_id`, `ad_id`, and `interaction_timestamps` don't help our models.
2. **Keep TF-IDF:** We are leaving the `tfidf` columns untouched because they represent the actual text content of the ads, which is crucial for predicting engagement.
3. **Yeo-Johnson Transformation:** Because the distribution of `clicks` is exponential, we apply Yeo-Johnson transformation to make the distribution more normal. This will allow us to get better results for our SVM model. The Yeo-Johnson distribution is good for exponential distributions and works with 0 and negative values.
4. **Normalization:** Apply Min-Max scaling to the continuous features.
5. **One-Hot Encoding:** Because the numbers assigned to the categorical features are arbitrary, we use one-hot encoding to prevent bias in our models.

In [ ]:
# 1. Load the original dataset
df = pd.read_csv('ad_campaign_data.csv')

# 2. Drop unnecessary identifiers and timestamps
columns_to_drop = ['user_id', 'ad_id', 'interaction_timestamps']
df_processed = df.drop(columns=columns_to_drop)

# 3. Yeo-Johnson transformation for SVMs
df_transformed = df.copy()
# Drop columns we're not using
df_transformed.drop(columns=columns_to_drop, inplace=True)
# Apply Yeo-Johnson transformation to clicks to reduce skew
transformer = PowerTransformer(method='yeo-johnson')
df_transformed['clicks'] = transformer.fit_transform(df_transformed[['clicks']])
# Visualize change in distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
# Clicks before transformation
axes[0].hist(df_processed['clicks'], bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Clicks (Raw)")
axes[0].set_xlabel("Clicks")
# Clicks after transformation
axes[1].hist(df_transformed['clicks'], bins=50, color="darkorange", edgecolor="white")
axes[1].set_title("Clicks (Yeo-Johnson Transformed)")
axes[1].set_xlabel("Clicks (Transformed)")
plt.tight_layout()
plt.show()

# 4. Normalization
continuous_features = ['age', 'impressions', 'clicks', 'previous_interaction_score', 'sentiment_score']
scaler = MinMaxScaler()
df_processed[continuous_features] = scaler.fit_transform(df_processed[continuous_features])
df_transformed[continuous_features] = scaler.fit_transform(df_transformed[continuous_features]) # for SVMs

# 5. One-Hot Encoding
encode_vars = ['gender', 'location', 'device_type', 'ad_category']
df_encoded = pd.get_dummies(df_processed, columns=encode_vars, prefix=encode_vars, dtype=int)
# For SVM model (with transformed clicks)
df_encoded_svm = pd.get_dummies(df_transformed, columns=encode_vars, prefix=encode_vars, dtype=int)

# 6. Move the target variable ('conversions') to the end
encoded_feature_cols = [col for col in df_encoded.columns if col != 'conversions']
df_encoded = df_encoded[encoded_feature_cols + ['conversions']]
df_encoded_svm = df_encoded_svm[encoded_feature_cols + ['conversions']]

# 7. Save to a clean CSV for the team to use
output_filename = 'final_project_data.csv'
df_encoded.to_csv(output_filename, index=False)
df_encoded_svm.to_csv('svm_project_data.csv', index=False)

print(f"Final dataset shape: {df_encoded.shape}")
df_encoded.head()

## Exploratory Data Analysis (EDA)
Let's visualize the distributions of our continuous and categorical features to understand the shape of our data before feeding it into our models.

In [ ]:
# Set up the visual style
sns.set_theme(style="whitegrid")

# Define our feature groups
continuous_features = ['age', 'impressions', 'clicks', 'engagement_duration', 'previous_interaction_score', 'sentiment_score']
categorical_features = ['gender', 'location', 'device_type', 'ad_category', 'conversions']

# --- Plot 1: Continuous Distributions ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distributions of Continuous Features', fontsize=16)
axes = axes.flatten()

for i, col in enumerate(continuous_features):
    sns.histplot(df[col], kde=True, ax=axes[i], bins=30, color='skyblue')
    axes[i].set_title(col)
    axes[i].set_xlabel('')

plt.tight_layout()
plt.show()

# --- Plot 2: Categorical Distributions ---
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distributions of Categorical Features', fontsize=16)
axes = axes.flatten()

for i, col in enumerate(categorical_features):
    val_counts = df[col].value_counts().sort_index()
    sns.barplot(x=val_counts.index, y=val_counts.values, hue=val_counts.index, ax=axes[i], palette='Set2', legend=False)
    axes[i].set_title(col)
    axes[i].set_xlabel('Category')
    axes[i].set_ylabel('Count')

axes[5].set_visible(False) # Hide the last empty subplot
plt.tight_layout()
plt.show()

# --- Plot 3: Correlation Matrix for Continuous Features ---
continuous_df = df_processed[continuous_features] # dataframe with continuous features
sns.heatmap(continuous_df.corr(), annot=True, vmin=-1, vmax=1, center=0, cmap='coolwarm', linewidths=0.5, linecolor='black', square=True)
plt.title('Correlation Matrix for Continuous Features', fontsize=16)
plt.show()

# --- Plot 4: Pairplot ---
pairplot_features = continuous_features
pairplot_features.append('conversions')
pairplot_df = df_processed[pairplot_features]
sns.set_style('whitegrid')
sns.pairplot(pairplot_df, hue='conversions')
plt.show()

## SVMs

#### Split Training and Testing Data

In [ ]:
# Read in the cleaned CSV file
df = pd.read_csv('final_project_data.csv')

# Separate independent and dependent variables
x = df.iloc[:, :-1]
y = df.loc[:, ['conversions']]

# Split the training and testing data with a ratio of 80:20
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

# Convert dependent variable to 1D array to work with sklearn
# y_train = y_train.to_numpy().ravel()
y_test = y_test.to_numpy().ravel()

#### SVM with Linear Kernel

In [ ]:
# # Create SVM classifier with RBF kernel
# model = svm.SVC(kernel='linear')
#
# # Specify hyperparameters to tune
# param_dict = {'C': [0.1, 1, 10, 100, 1000],
#               'gamma': [1, 0.1, 0.01, 0.001, 0.0001]}
#
# # Perform hyperparameter tuning with random search
# random_search = RandomizedSearchCV(model, param_distributions=param_dict, n_iter=10, cv=5, verbose=2, n_jobs=-1)
# random_search.fit(x_train, y_train)
#
# # Get best model
# best_linear_model = random_search.best_estimator_
#
# Save best model
# joblib.dump(best_linear_model, 'linear_svm.joblib')

# Load best model
linear_svm = joblib.load('linear_svm.joblib')

# Make predictions on testing data with best hyperparameters
y_pred_linear = linear_svm.predict(x_test)

# Display best hyperparameters
print('--- SVM with Linear Kernel ---')
print(f'Best Hyperparameters:\nC: {linear_svm.C}\ngamma: {linear_svm.gamma}')

# Display metrics on the best linear model
class_report = classification_report(y_test, y_pred_linear, target_names=['Not Converted (0)', 'Converted (1)'], digits=4)
print(class_report)

# Create confusion matrix for the testing set
matrix_labels = ['Not Converted (0)', 'Converted (1)']
matrix = confusion_matrix(y_test, y_pred_linear)
ax = plt.subplot()
sns.heatmap(matrix, annot=True, fmt="g", ax=ax, cmap="flare",
            linewidths=0.5, linecolor="black", square=True)
ax.set_title("Testing Set Confusion Matrix for SVM with Linear Kernel", fontsize=20)
ax.set_xlabel("Predicted Labels", fontsize=12)
ax.set_ylabel("True Labels", fontsize=12)
ax.xaxis.set_ticklabels(matrix_labels, fontsize=12)
ax.yaxis.set_ticklabels(matrix_labels, fontsize=12)
plt.show()

#### SVM with RBF Kernel

In [ ]:
# # Create SVM classifier with RBF kernel
# rbf_model = svm.SVC(kernel='rbf')
#
# # Specify hyperparameters to tune
# param_dict = {'C': [0.1, 1, 10, 100, 1000],
#               'gamma': [1, 0.1, 0.01, 0.001, 0.0001]}
#
# # Perform hyperparameter tuning with random search
# random_search_rbf = RandomizedSearchCV(rbf_model, param_distributions=param_dict, n_iter=10, cv=5, verbose=2, n_jobs=-1)
# random_search_rbf.fit(x_train, y_train)
#
# # Get best model
# best_rbf_model = random_search_rbf.best_estimator_
#
# # Save best model
# joblib.dump(best_rbf_model, 'rbf_svm.joblib')

# Load best model
rbf_svm = joblib.load('rbf_svm.joblib')

# Make predictions on testing data
y_pred_rbf = rbf_svm.predict(x_test)

# Display best hyperparameters
print('--- SVM with RBF Kernel ---')
print(f'Best Hyperparameters:\nC: {rbf_svm.C}\ngamma: {rbf_svm.gamma}')

# Display metrics on the best RBF model
class_report_rbf = classification_report(y_test, y_pred_rbf, target_names=['Not Converted (0)', 'Converted (1)'], digits=4)
print(class_report_rbf)

# Create confusion matrix for the testing set
matrix_labels = ['Not Converted (0)', 'Converted (1)']
rbf_matrix = confusion_matrix(y_test, y_pred_rbf)
ax = plt.subplot()
sns.heatmap(rbf_matrix, annot=True, fmt="g", ax=ax, cmap="flare",
            linewidths=0.5, linecolor="black", square=True)
ax.set_title("Testing Set Confusion Matrix for SVM with RBF Kernel", fontsize=20)
ax.set_xlabel("Predicted Labels", fontsize=12)
ax.set_ylabel("True Labels", fontsize=12)
ax.xaxis.set_ticklabels(matrix_labels, fontsize=12)
ax.yaxis.set_ticklabels(matrix_labels, fontsize=12)
plt.show()

#### Random Oversampling

In [ ]:
# Use random oversampling
oversample = RandomOverSampler(random_state=0)

# Apply oversampling to the data
x_over, y_over = oversample.fit_resample(x_train, y_train)

# Summarize the 'conversions' class distribution
distribution = dict(Counter(y_train.to_numpy().ravel()))
over_distribution = dict(Counter(y_over.to_numpy().ravel()))
print(f'Class Distribution Before Oversampling:\n1: {distribution[1]}, 0: {distribution[0]}')
print(f'\nClass Distribution After Oversampling:\n1: {over_distribution[1]}, 0: {over_distribution[0]}')

#### SVM with Linear Kernel and Random Oversampling

In [ ]:
# # Create SVM classifier with linear kernel
# model = svm.SVC(kernel='linear')
#
# # Specify hyperparameters to tune
# param_dict = {'C': [0.1, 1, 10, 100, 1000],
#               'gamma': [1, 0.1, 0.01, 0.001, 0.0001]}
#
# # Perform hyperparameter tuning with random search
# random_search = RandomizedSearchCV(model, param_distributions=param_dict, n_iter=10, cv=5, verbose=2, n_jobs=-1)
# random_search.fit(x_over, y_over)
#
# # Get best model
# best_linear_model = random_search.best_estimator_
#
# # Save best model
# joblib.dump(best_linear_model, 'oversampling_linear_svm.joblib')

# Load best model
oversampling_linear_svm = joblib.load('oversampling_linear_svm.joblib')

# Make predictions on testing data with best hyperparameters
y_pred_linear = oversampling_linear_svm.predict(x_test)

# Display best hyperparameters
print('--- SVM with Linear Kernel and Random Oversampling---')
print(f'Best Hyperparameters:\nC: {oversampling_linear_svm.C}\ngamma: {oversampling_linear_svm.gamma}')

# Display metrics on the best linear model
class_report = classification_report(y_test, y_pred_linear, target_names=['Not Converted (0)', 'Converted (1)'], digits=4)
print(class_report)

# Create confusion matrix for the testing set
matrix_labels = ['Not Converted (0)', 'Converted (1)']
matrix = confusion_matrix(y_test, y_pred_linear)
ax = plt.subplot()
sns.heatmap(matrix, annot=True, fmt="g", ax=ax, cmap="flare",
            linewidths=0.5, linecolor="black", square=True)
ax.set_title("Testing Set Confusion Matrix for SVM with Linear Kernel", fontsize=20)
ax.set_xlabel("Predicted Labels", fontsize=12)
ax.set_ylabel("True Labels", fontsize=12)
ax.xaxis.set_ticklabels(matrix_labels, fontsize=12)
ax.yaxis.set_ticklabels(matrix_labels, fontsize=12)
plt.show()

#### SVM with RBF Kernel and Random Oversampling

In [ ]:
# # Create SVM classifier with RBF kernel
# rbf_model = svm.SVC(kernel='rbf')
#
# # Specify hyperparameters to tune
# param_dict = {'C': [0.1, 1, 10, 100, 1000],
#               'gamma': [1, 0.1, 0.01, 0.001, 0.0001]}
#
# # Perform hyperparameter tuning with random search
# random_search_rbf = RandomizedSearchCV(rbf_model, param_distributions=param_dict, n_iter=10, cv=5, verbose=2, n_jobs=-1)
# random_search_rbf.fit(x_over, y_over)
#
# # Get best model
# best_rbf_model = random_search_rbf.best_estimator_
#
# # Save best model
# joblib.dump(best_rbf_model, 'oversampling_rbf_svm.joblib')

# Load best model
oversampling_rbf_svm = joblib.load('oversampling_rbf_svm.joblib')

# Make predictions on the testing set
y_pred_rbf = oversampling_rbf_svm.predict(x_test)

# Display best hyperparameters
print('--- SVM with RBF Kernel and Random Oversampling---')
print(f'Best Hyperparameters:\nC: {oversampling_rbf_svm.C}\ngamma: {oversampling_rbf_svm.gamma}')

# Display metrics on the best RBF model
class_report_rbf = classification_report(y_test, y_pred_rbf, target_names=['Not Converted (0)', 'Converted (1)'], digits=4)
print(class_report_rbf)

# Create confusion matrix for the testing set
matrix_labels = ['Not Converted (0)', 'Converted (1)']
rbf_matrix = confusion_matrix(y_test, y_pred_rbf)
ax = plt.subplot()
sns.heatmap(rbf_matrix, annot=True, fmt="g", ax=ax, cmap="flare",
            linewidths=0.5, linecolor="black", square=True)
ax.set_title("Testing Set Confusion Matrix for SVM with RBF Kernel", fontsize=20)
ax.set_xlabel("Predicted Labels", fontsize=12)
ax.set_ylabel("True Labels", fontsize=12)
ax.xaxis.set_ticklabels(matrix_labels, fontsize=12)
ax.yaxis.set_ticklabels(matrix_labels, fontsize=12)
plt.show()

#### Downsampling

In [ ]:
# Show how many instances there are of the 0 class in the training data
n = x_train[y_train['conversions'] == 0].shape[0]
print(f'Number of Samples with Minority Class: {n}')

# Downsampling: Randomly sample n instances from both class 0 and class 1 of training data
# n is number of instances of the minority class in the training data
# Setting random_state ensures reproducibility for everyone in the group
df_class_0 = x_train[y_train['conversions'] == 0].sample(n=n, random_state=42)

df_class_1 = x_train[y_train['conversions'] == 1].sample(n=n, random_state=42)

# Combine the balanced classes
x_balanced = pd.concat([df_class_0, df_class_1])

# Get correct indices for 'conversions' for training set
y_train = y_train.loc[x_balanced.index]

#### SVM with Linear Kernel and Downsampling

In [ ]:
# # Create SVM classifier with linear kernel
# model = svm.SVC(kernel='linear')
#
# # Specify hyperparameters to tune
# param_dict = {'C': [0.1, 1, 10, 100, 1000],
#               'gamma': [1, 0.1, 0.01, 0.001, 0.0001]}
#
# # Perform hyperparameter tuning with random search
# random_search = RandomizedSearchCV(model, param_distributions=param_dict, n_iter=10, cv=5, verbose=2, n_jobs=-1)
# random_search.fit(x_balanced, y_train)
#
# # Get best model
# best_linear_model = random_search.best_estimator_
#
# # Save best model
# joblib.dump(best_linear_model, 'downsampling_linear_svm.joblib')

# Load best model
downsampling_linear_svm = joblib.load('downsampling_linear_svm.joblib')

# Make predictions on training data
y_pred_linear = downsampling_linear_svm.predict(x_test)

# Display best hyperparameters
print('--- SVM with Linear Kernel and Downsampling ---')
print(f'Best Hyperparameters:\nC: {downsampling_linear_svm.C}\ngamma: {downsampling_linear_svm.gamma}')

# Display metrics on the best linear model
class_report = classification_report(y_test, y_pred_linear, target_names=['Not Converted (0)', 'Converted (1)'], digits=4)
print(class_report)

# Create confusion matrix for the testing set
matrix_labels = ['Not Converted (0)', 'Converted (1)']
matrix = confusion_matrix(y_test, y_pred_linear)
ax = plt.subplot()
sns.heatmap(matrix, annot=True, fmt="g", ax=ax, cmap="flare",
            linewidths=0.5, linecolor="black", square=True)
ax.set_title("Testing Set Confusion Matrix for SVM with Linear Kernel", fontsize=20)
ax.set_xlabel("Predicted Labels", fontsize=12)
ax.set_ylabel("True Labels", fontsize=12)
ax.xaxis.set_ticklabels(matrix_labels, fontsize=12)
ax.yaxis.set_ticklabels(matrix_labels, fontsize=12)
plt.show()

#### SVM with RBF Kernel and Downsampling

In [ ]:
# # Create SVM classifier with RBF kernel
# rbf_model = svm.SVC(kernel='rbf')
#
# # Specify hyperparameters to tune
# param_dict = {'C': [0.1, 1, 10, 100, 1000],
#               'gamma': [1, 0.1, 0.01, 0.001, 0.0001]}
#
# # Perform hyperparameter tuning with random search
# random_search_rbf = RandomizedSearchCV(rbf_model, param_distributions=param_dict, n_iter=10, cv=5, verbose=2, n_jobs=-1)
# random_search_rbf.fit(x_balanced, y_train)
#
# # Get best model
# best_rbf_model = random_search_rbf.best_estimator_
#
# # Save best model
# joblib.dump(best_rbf_model, 'downsampling_rbf_svm.joblib')

# Load best model
downsampling_rbf_svm = joblib.load('downsampling_rbf_svm.joblib')

# Make predictions on testing set
y_pred_rbf = downsampling_rbf_svm.predict(x_test)

# Display best hyperparameters
print('--- SVM with RBF Kernel and Downsampling ---')
print(f'Best Hyperparameters:\nC: {downsampling_rbf_svm.C}\ngamma: {downsampling_rbf_svm.gamma}')

# Display metrics on the best RBF model
class_report_rbf = classification_report(y_test, y_pred_rbf, target_names=['Not Converted (0)', 'Converted (1)'], digits=4)
print(class_report_rbf)

# Create confusion matrix for the testing set
matrix_labels = ['Not Converted (0)', 'Converted (1)']
rbf_matrix = confusion_matrix(y_test, y_pred_rbf)
ax = plt.subplot()
sns.heatmap(rbf_matrix, annot=True, fmt="g", ax=ax, cmap="flare",
            linewidths=0.5, linecolor="black", square=True)
ax.set_title("Testing Set Confusion Matrix for SVM with RBF Kernel", fontsize=20)
ax.set_xlabel("Predicted Labels", fontsize=12)
ax.set_ylabel("True Labels", fontsize=12)
ax.xaxis.set_ticklabels(matrix_labels, fontsize=12)
ax.yaxis.set_ticklabels(matrix_labels, fontsize=12)
plt.show()

#### Transformed 'Clicks' with Yeo-Johnson Transformation
The Yeo-Johnson transformation was used to make the distribution of `clicks` more normal.

In [ ]:
# Read in the cleaned CSV file
transformed_df = pd.read_csv('svm_project_data.csv')

# Separate independent and dependent variables
x = transformed_df.iloc[:, :-1]
y = transformed_df.loc[:, ['conversions']]

# Split the training and testing data with a ratio of 80:20
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=0)

# Convert dependent variable to 1D array to work with sklearn
# y_train = y_train.to_numpy().ravel()
y_test = y_test.to_numpy().ravel()

#### SVM with Linear Kernel and Transformed `Clicks`

In [ ]:
# # Create SVM classifier with linear kernel
# model = svm.SVC(kernel='linear')
#
# # Specify hyperparameters to tune
# param_dict = {'C': [0.1, 1, 10, 100, 1000],
#               'gamma': [1, 0.1, 0.01, 0.001, 0.0001]}
#
# # Perform hyperparameter tuning with random search
# random_search = RandomizedSearchCV(model, param_distributions=param_dict, n_iter=10, cv=5, verbose=2, n_jobs=-1)
# random_search.fit(x_train, y_train)
#
# # Get best model
# best_linear_model = random_search.best_estimator_
#
# # Save best model
# joblib.dump(best_linear_model, 'transformed_clicks_linear_svm.joblib')

# Load best model
transformed_linear_svm = joblib.load('transformed_clicks_linear_svm.joblib')

# Make predictions on testing set
y_pred_linear = transformed_linear_svm.predict(x_test)

# Display best hyperparameters
print("--- SVM with Linear Kernel and Transformed 'Clicks' ---")
print(f'Best Hyperparameters:\nC: {transformed_linear_svm.C}\ngamma: {transformed_linear_svm.gamma}')

# Display metrics on the best linear model
class_report = classification_report(y_test, y_pred_linear, target_names=['Not Converted (0)', 'Converted (1)'], digits=4)
print(class_report)

# Create confusion matrix for the testing set
matrix_labels = ['Not Converted (0)', 'Converted (1)']
matrix = confusion_matrix(y_test, y_pred_linear)
ax = plt.subplot()
sns.heatmap(matrix, annot=True, fmt="g", ax=ax, cmap="flare",
            linewidths=0.5, linecolor="black", square=True)
ax.set_title("Testing Set Confusion Matrix for SVM with Linear Kernel", fontsize=20)
ax.set_xlabel("Predicted Labels", fontsize=12)
ax.set_ylabel("True Labels", fontsize=12)
ax.xaxis.set_ticklabels(matrix_labels, fontsize=12)
ax.yaxis.set_ticklabels(matrix_labels, fontsize=12)
plt.show()

#### SVM with RBF Kernel and Transformed `Clicks`

In [ ]:
# # Create SVM classifier with RBF kernel
# rbf_model = svm.SVC(kernel='rbf')
#
# # Specify hyperparameters to tune
# param_dict = {'C': [0.1, 1, 10, 100, 1000],
#               'gamma': [1, 0.1, 0.01, 0.001, 0.0001]}
#
# # Perform hyperparameter tuning with random search
# random_search_rbf = RandomizedSearchCV(rbf_model, param_distributions=param_dict, n_iter=10, cv=5, verbose=2, n_jobs=-1)
# random_search_rbf.fit(x_train, y_train)
#
# # Get best model
# best_rbf_model = random_search_rbf.best_estimator_
#
# # Save best model
# joblib.dump(best_rbf_model, 'transformed_clicks_rbf_svm.joblib')

# Load best model
transformed_rbf_svm = joblib.load('transformed_clicks_rbf_svm.joblib')

# Make predictions on testing data with best hyperparameters
y_pred_rbf = transformed_rbf_svm.predict(x_test)

# Display best hyperparameters
print("--- SVM with RBF Kernel and Transformed 'Clicks' ---")
print(f'Best Hyperparameters:\nC: {transformed_rbf_svm.C}\ngamma: {transformed_rbf_svm.gamma}')

# Display metrics on the best RBF model
class_report_rbf = classification_report(y_test, y_pred_rbf, target_names=['Not Converted (0)', 'Converted (1)'], digits=4)
print(class_report_rbf)

# Create confusion matrix for the testing set
matrix_labels = ['Not Converted (0)', 'Converted (1)']
rbf_matrix = confusion_matrix(y_test, y_pred_rbf)
ax = plt.subplot()
sns.heatmap(rbf_matrix, annot=True, fmt="g", ax=ax, cmap="flare",
            linewidths=0.5, linecolor="black", square=True)
ax.set_title("Testing Set Confusion Matrix for SVM with RBF Kernel", fontsize=20)
ax.set_xlabel("Predicted Labels", fontsize=12)
ax.set_ylabel("True Labels", fontsize=12)
ax.xaxis.set_ticklabels(matrix_labels, fontsize=12)
ax.yaxis.set_ticklabels(matrix_labels, fontsize=12)
plt.show()

## Hidden Layer Models 1-2

In [3]:
#load/prep data
df = pd.read_csv('final_project_data.csv')

#get cols
x_cols = df.columns[:-1]
y_col = df.columns[-1]

#get data
x = df[x_cols].values.astype(np.float64)
y = df[y_col].values.astype(int)

#split into train and test
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, 
random_state=42)

#save some for validation
x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.2, 
random_state=42)

In [4]:
#setting up MLP

#make MLP
def build_clf(hidden1 = 32, rate = 0.01, dropout_rate = 0, **kwargs):
    model = Sequential([
        Input(shape=(x.shape[1],)),
        Dense(hidden1, activation = 'relu'),
        Dropout(dropout_rate),
        Dense(1, activation = 'sigmoid')
    ])

#https://www.tensorflow.org/api_docs/python/tf/keras/Model
    
    model.compile(optimizer=SGD(learning_rate = rate), loss = 'binary_crossentropy',
                  metrics = ['accuracy'])
    return model

model = KerasClassifier(model=build_clf, epochs = 50, batch_size = 128, verbose = 1)

#test diff params
params = {
    'model__hidden1':[32, 64, 128],
    'model__rate':[0.0001, 0.0005, 0.001, 0.005, 0.01],
    'epochs': [50, 100, 150],
    'model__dropout_rate': [0, 0.1, 0.3, 0.5]
}
#class sklearn.model_selection.RandomizedSearchCV(estimator, param_distributions, 
#*, n_iter=10, scoring=None, n_jobs=None, refit=True, cv=None, verbose=0, 
#pre_dispatch='2*n_jobs', random_state=None, error_score=nan, return_train_score=False)

#using random search 
random_search = RandomizedSearchCV(estimator = model, param_distributions = params,
                    n_iter = 25, scoring = 'f1', cv = 3, verbose = 1,
                    random_state = 42)

In [5]:
#fitting

# #fit on training data
#original_rs = random_search.fit(x_train,y_train)

original_rs = joblib.load("1layeroriginal.pkl")

print("Best CV F1: ", original_rs.best_score_)
print("Best Params: ", original_rs.best_params_)

Best CV F1:  0.9324474503088566
Best Params:  {'model__rate': 0.01, 'model__hidden1': 32, 'model__dropout_rate': 0.1, 'epochs': 50}


In [1]:
np.random.seed(42)

original_best_model = original_rs.best_estimator_

y_pred = original_best_model.predict(x_test)

#full confusion matrix printed below graph
cm = confusion_matrix(y_test, y_pred)

print("Confusion matrix:")
print(cm)

f1 = f1_score(y_test, y_pred)

print("F1 Score: ", f1)
print("Classification")
print(classification_report(y_test,y_pred))

NameError: name 'np' is not defined

In [ ]:
#make class weights
np.random.seed(42)
random.seed(42)

classes = np.array([0,1])
weights = compute_class_weight(class_weight="balanced", classes = classes, y = y_train)

weight_class = {0: weights[0], 1: weights[1]}
print(weight_class)

In [ ]:
#fitting with new weights

#fit on training data
#rs = random_search.fit(x_train,y_train, class_weight=weight_class)

rs = joblib.load("1layerweight.pkl")

print("Best CV F1: ", rs.best_score_)
print("Best Params: ", rs.best_params_)

In [ ]:
#final best model + f1 scores eval

#use test data to get f1 stats

np.random.seed(42)
random.seed(42)

best_model = rs.best_estimator_

y_pred_weighed = best_model.predict(x_test)

#full confusion matrix printed below graph
cm_1 = confusion_matrix(y_test, y_pred_weighed)

print("Confusion matrix:")
print(cm_1)

f1_1 = f1_score(y_test, y_pred_weighed)

print("F1 Score: ", f1_1)
print("Classification")
print(classification_report(y_test,y_pred_weighed))

In [ ]:
plt.figure(figsize=(6,4))

sns.heatmap(
    cm_1, 
    annot = True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Not Converted(0)', 'Converted(1)'],
    yticklabels=['Not Converted (0)', 'Converted (1)'],
)

plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Orignal 1 Hidden Layer CM')

In [ ]:
#threshold tuning for 1 layer

y_prob = best_model.predict_proba(x_val)

thresholds = np.arange(0.1, 0.5, 0.01)

best_threshold = 0.5
y_pred_best = (y_prob[:,1] > 0.5).astype(int)
best_f1 = f1_score(y_val, y_pred_best)

print(y_pred_best)
for i in thresholds :
    temp = (y_prob[:,1] > i).astype(int)
    
    score = f1_score(y_val, temp)
    if score > best_f1:
        best_f1 = score
        best_threshold = i
        y_pred_best = temp

cm_best = confusion_matrix(y_val, y_pred_best)

print("Confusion matrix:")
print(cm_best)

print(classification_report(y_val,y_pred_best))

print("F1 Score: ", best_f1)
print("Threshold: ", best_threshold)

In [ ]:
y_prob_test = best_model.predict_proba(x_test)
y_pred_test = (y_prob_test[:,1] > best_threshold).astype(int)

cm_test = confusion_matrix(y_test, y_pred_test)
f1_test = f1_score(y_test, y_pred_test)

print("Confusion matrix:")
print(cm_test)

print(classification_report(y_test,y_pred_test))

print("F1 Score: ", f1_test)


In [ ]:
plt.figure(figsize=(6,4))

sns.heatmap(
    cm_test, 
    annot = True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Not Converted(0)', 'Converted(1)'],
    yticklabels=['Not Converted (0)', 'Converted (1)'],
)

plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('0.3 Threshold 1 Hidden Layer CM')

In [ ]:
#2 layers
#make MLP
def build_clf2(hidden1 = 32, hidden2 = 32, rate = 0.01, dropout_rate = 0, **kwargs):
    model2 = Sequential([
        Input(shape=(x.shape[1],)),
        Dense(hidden1, activation = 'relu'),
        Dense(hidden2, activation = 'relu'),
        Dropout(dropout_rate),
        Dense(1, activation = 'sigmoid')
    ])

#https://www.tensorflow.org/api_docs/python/tf/keras/Model
    
    model2.compile(optimizer=SGD(learning_rate = rate), loss = 'binary_crossentropy',
                  metrics = ['accuracy'])
    return model2

model2 = KerasClassifier(model=build_clf2, epochs = 10, batch_size = 128, verbose = 1)

#test diff params
params2 = {
    'model__hidden1':[32, 64, 128],
    'model__hidden2':[32, 64, 128],
    'model__rate':[0.0005, 0.001, 0.005],
    'epochs': [50, 100, 150],
    'model__dropout_rate': [0, 0.1, 0.3, 0.5]
}
#class sklearn.model_selection.RandomizedSearchCV(estimator, param_distributions, 
#*, n_iter=10, scoring=None, n_jobs=None, refit=True, cv=None, verbose=0, 
#pre_dispatch='2*n_jobs', random_state=None, error_score=nan, return_train_score=False)

#using random search 
random_search2 = RandomizedSearchCV(estimator = model2, param_distributions = params2,
                    n_iter = 50, scoring = 'f1', cv = 3, verbose = 1, 
                    random_state = 42)


In [ ]:
#fitting layer 2

#fit on training data
#rs2 = random_search2.fit(x_train,y_train, class_weight=weight_class)
rs2 = joblib.load("2layer.pkl")

print("Best CV F1: ", rs2.best_score_)
print("Best Params: ", rs2.best_params_)

In [ ]:
#final best model + f1 scores eval

#use test data to get f1 stats

np.random.seed(42)
random.seed(42)
best_model2 = rs2.best_estimator_

#so we can see error of each epoch
#history2refine = best_model2refine.fit(x_train, y_train,
                    #validation_data = (x_test, y_test), class_weight = weight_class, verbose = 1)

y_pred2 = best_model2.predict(x_test)

#full confusion matrix printed below graph
cm2 = confusion_matrix(y_test, y_pred2)

print("Confusion matrix:")
print(cm2)

f1_2 = f1_score(y_test, y_pred2)
print(classification_report(y_test,y_pred2))

print("F1 Score: ", f1_2)


In [ ]:
plt.figure(figsize=(6,4))

sns.heatmap(
    cm2, 
    annot = True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Not Converted(0)', 'Converted(1)'],
    yticklabels=['Not Converted (0)', 'Converted (1)'],
)

plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('2 Hidden Layer CM')

In [ ]:
y_prob2 = best_model2.predict_proba(x_val)

thresholds = np.arange(0.1, 0.9, 0.01)

best_threshold2 = 0.5
y_pred2_best = (y_prob2[:,1] > 0.5).astype(int)
best_f1_2 = f1_score(y_val, y_pred2_best)

for i in thresholds :
    temp = (y_prob2[:,1] > i).astype(int)
    
    score = f1_score(y_val, temp)
    if score > best_f1_2:
        best_f1_2 = score
        best_threshold2 = i
        y_pred2_best = temp

cm2_best = confusion_matrix(y_val, y_pred2_best)

print("Confusion matrix:")
print(cm2_best)

print("F1 Score: ", best_f1_2)
print("Threshold: ", best_threshold2)

In [ ]:
y_prob2_test = best_model2.predict_proba(x_test)
y_pred2_test = (y_prob2_test[:,1] > best_threshold2).astype(int)

cm2_test = confusion_matrix(y_test, y_pred2_test)
f1_test2 = f1_score(y_test, y_pred2_test)

print("Confusion matrix:")
print(cm2_test)

print(classification_report(y_test,y_pred2_test))

print("F1 Score: ", f1_test2)


In [ ]:
plt.figure(figsize=(6,4))

sns.heatmap(
    cm2_test, 
    annot = True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Not Converted(0)', 'Converted(1)'],
    yticklabels=['Not Converted (0)', 'Converted (1)'],
)

plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('2 Hidden Layer CM')

## Hidden Layer Models 3-4

In [ ]:
def create_mlp_model(num_hidden_layers=2, learning_rate=0.01, momentum=0.0,
                     l2_lambda=0.0, dropout_rate=0.0,
                     num_hidden_units_1=128, num_hidden_units_2=64, num_hidden_units_3=32, num_hidden_units_4=16):
    model = Sequential()

    model.add(Dense(num_hidden_units_1, activation='relu', input_shape=(num_features,),
                    kernel_regularizer=l2(l2_lambda)))
    model.add(Dropout(dropout_rate))

    model.add(Dense(num_hidden_units_2, activation='relu', kernel_regularizer=l2(l2_lambda)))
    model.add(Dropout(dropout_rate))

    if num_hidden_layers >= 3:
        model.add(Dense(num_hidden_units_3, activation='relu', kernel_regularizer=l2(l2_lambda)))
        model.add(Dropout(dropout_rate))

    if num_hidden_layers == 4:
        model.add(Dense(num_hidden_units_4, activation='relu', kernel_regularizer=l2(l2_lambda)))
        model.add(Dropout(dropout_rate))

    model.add(Dense(1, activation='sigmoid'))

    optimizer = SGD(learning_rate=learning_rate, momentum=momentum)

    model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
class_weights = class_weight.compute_class_weight('balanced',
                                                  classes=np.unique(y_train),
                                                  y=y_train.ravel())

class_weights_dict = dict(enumerate(class_weights))

print("Calculated Class Weights:", class_weights_dict)

In [ ]:
final_params_3_layer = {
    'model__num_hidden_units_3': 16,
    'model__num_hidden_units_2': 32,
    'model__num_hidden_units_1': 64,
    'model__num_hidden_layers': 3,
    'model__momentum': 0.0,
    'model__learning_rate': 0.001,
    'model__l2_lambda': 0.0,
    'model__dropout_rate': 0.1,
    'epochs': 200,
    'batch_size': 64
}

print("\n--- Training Final 3-Hidden Layer Weighted MLP Model ---")

final_mlp_3_layer_weighted = KerasClassifier(model=create_mlp_model,
                                             num_hidden_layers=final_params_3_layer['model__num_hidden_layers'],
                                             learning_rate=final_params_3_layer['model__learning_rate'],
                                             momentum=final_params_3_layer['model__momentum'],
                                             l2_lambda=final_params_3_layer['model__l2_lambda'],
                                             dropout_rate=final_params_3_layer['model__dropout_rate'],
                                             num_hidden_units_1=final_params_3_layer['model__num_hidden_units_1'],
                                             num_hidden_units_2=final_params_3_layer['model__num_hidden_units_2'],
                                             num_hidden_units_3=final_params_3_layer['model__num_hidden_units_3'],
                                             epochs=final_params_3_layer['epochs'],
                                             batch_size=final_params_3_layer['batch_size'],
                                             verbose=0)

final_mlp_3_layer_weighted.fit(X_train, y_train.ravel(), class_weight=class_weights_dict)

final_model_3_layer_path = 'final_mlp_3_layer_weighted.keras'
final_mlp_3_layer_weighted.model_.save(final_model_3_layer_path)
print(f"Final 3-Hidden Layer Weighted MLP Model saved to: {final_model_3_layer_path}")

In [ ]:
weights_path_3_layer = 'final_mlp_3_layer_weighted_weights.weights.h5'
final_mlp_3_layer_weighted.model_.save_weights(weights_path_3_layer)
print(f"Final 3-Hidden Layer Weighted MLP Model weights saved to: {weights_path_3_layer}")

In [ ]:
final_params_4_layer = {
    'model__num_hidden_units_3': 16,
    'model__num_hidden_units_2': 32,
    'model__num_hidden_units_1': 64,
    'model__num_hidden_layers': 4,
    'model__num_hidden_units_4': 8,
    'model__momentum': 0.0,
    'model__learning_rate': 0.001,
    'model__l2_lambda': 0.0,
    'model__dropout_rate': 0.1,
    'epochs': 200,
    'batch_size': 64
}

print("\n--- Training Final 4-Hidden Layer Weighted MLP Model ---")

final_mlp_4_layer_weighted = KerasClassifier(model=create_mlp_model,
                                             num_hidden_layers=final_params_4_layer['model__num_hidden_layers'],
                                             learning_rate=final_params_4_layer['model__learning_rate'],
                                             momentum=final_params_4_layer['model__momentum'],
                                             l2_lambda=final_params_4_layer['model__l2_lambda'],
                                             dropout_rate=final_params_4_layer['model__dropout_rate'],
                                             num_hidden_units_1=final_params_4_layer['model__num_hidden_units_1'],
                                             num_hidden_units_2=final_params_4_layer['model__num_hidden_units_2'],
                                             num_hidden_units_3=final_params_4_layer['model__num_hidden_units_3'],
                                             num_hidden_units_4=final_params_4_layer['model__num_hidden_units_4'],
                                             epochs=final_params_4_layer['epochs'],
                                             batch_size=final_params_4_layer['batch_size'],
                                             verbose=0)

final_mlp_4_layer_weighted.fit(X_train, y_train.ravel(), class_weight=class_weights_dict)

final_model_4_layer_path = 'final_mlp_4_layer_weighted.keras'
final_mlp_4_layer_weighted.model_.save(final_model_4_layer_path)
print(f"Final 4-Hidden Layer Weighted MLP Model saved to: {final_model_4_layer_path}")

In [ ]:
weights_path_4_layer = 'final_mlp_4_layer_weighted_weights.weights.h5'
final_mlp_4_layer_weighted.model_.save_weights(weights_path_4_layer)
print(f"Final 4-Hidden Layer Weighted MLP Model weights saved to: {weights_path_4_layer}")

In [ ]:
from tensorflow.keras.models import load_model

loaded_mlp_3_layer = load_model('final_mlp_3_layer_weighted.keras', compile=False)
print("Loaded 3-Hidden Layer Model Summary:")
loaded_mlp_3_layer.summary()

loaded_scikeras_3_layer = KerasClassifier(model=create_mlp_model, model__num_hidden_layers=3,
                                          model__learning_rate=0.001, model__momentum=0.0,
                                          model__l2_lambda=0.0, model__dropout_rate=0.1,
                                          model__num_hidden_units_1=64, model__num_hidden_units_2=32,
                                          model__num_hidden_units_3=16, epochs=200, batch_size=64)
loaded_scikeras_3_layer.initialize(X=X_test, y=y_test)
loaded_scikeras_3_layer.set_params(model=loaded_mlp_3_layer)

loaded_mlp_4_layer = load_model('final_mlp_4_layer_weighted.keras', compile=False)
print("\nLoaded 4-Hidden Layer Model Summary:")
loaded_mlp_4_layer.summary()

loaded_scikeras_4_layer = KerasClassifier(model=create_mlp_model, model__num_hidden_layers=4,
                                          model__learning_rate=0.001, model__momentum=0.0,
                                          model__l2_lambda=0.0, dropout_rate=0.1,
                                          model__num_hidden_units_1=64, model__num_hidden_units_2=32,
                                          model__num_hidden_units_3=16, model__num_hidden_units_4=8,
                                          epochs=200, batch_size=64)
loaded_scikeras_4_layer.initialize(X=X_test, y=y_test)
loaded_scikeras_4_layer.set_params(model=loaded_mlp_4_layer)

print("\nModels loaded successfully. You can now use `loaded_scikeras_3_layer` and `loaded_scikeras_4_layer` for predictions.")